# Paper 3 — Reproduce Figure 7A and 7B
**Inference only**; this notebook never trains weights. Uses trained checkpoints and the official local dSprites file. Expect one complete dataset encoder sweep for each model.

In [ ]:
# Clone private source without embedding or printing the access token.
from pathlib import Path
import os, subprocess, tempfile, stat
from google.colab import userdata

REPO = Path('/content/advanced-ml-paper-reproductions')
PROJECT = REPO / 'paper3-beta-vae'
if not PROJECT.exists():
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Add GITHUB_TOKEN to Colab Secrets and grant notebook access.')
    with tempfile.TemporaryDirectory() as tmp:
        askpass = Path(tmp) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "x-access-token" ;;\n  *Password*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0')
        subprocess.run(['git','clone','--quiet', 'https://github.com/vepaan/advanced-ml-paper-reproductions.git',str(REPO)], env=env, check=True)
    del token
    print('Cloned private repository to', REPO)
else:
    print('Reusing existing project checkout:', PROJECT)
assert (PROJECT / 'src' / 'models.py').is_file(), 'Push paper3-beta-vae/ into GitHub first and rerun setup.'
%cd /content/advanced-ml-paper-reproductions/paper3-beta-vae
# Colab usually ships matplotlib. Install if missing; do NOT replace Colab torch with old Windows pins.
import importlib.util
if importlib.util.find_spec('matplotlib') is None:
    subprocess.run(['python','-m','pip','install','-q','matplotlib'],check=True)
import torch
print('torch:',torch.__version__,'CUDA:',torch.cuda.is_available(),
      'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
from IPython.display import display, Image

drive.mount('/content/drive')
FNAME='dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz'
DATA=Path('/content/datasets/dsprites')/FNAME
DATA.parent.mkdir(parents=True,exist_ok=True)
DATA_SRC=Path('/content/drive/MyDrive/ML-Paper-Reproductions/datasets/dsprites')/FNAME
if not DATA.exists():
    if not DATA_SRC.is_file(): raise FileNotFoundError(f'Upload dataset to {DATA_SRC}')
    shutil.copy2(DATA_SRC,DATA)
RUNS=Path('/content/paper3_runs')
BACKUP=Path('/content/drive/MyDrive/ML-Paper-Reproductions/paper3')
for beta in (1,4):
    saved=BACKUP/f'beta{beta}'/'model.pt'
    local=RUNS/f'beta{beta}'/'model.pt'
    local.parent.mkdir(parents=True,exist_ok=True)
    if not local.is_file():
        if not saved.is_file(): raise FileNotFoundError(f'Run training first; missing {saved}')
        shutil.copy2(saved,local)
    print('Model ready:',local)
FIGURES=Path('/content/paper3_figure7')
FIGURES.mkdir(parents=True,exist_ok=True)


## Figure 7A: β=4 and Figure 7B: β=1
Every plot uses actual inferred posterior **means**: 32×32 position maps, shape-wise scale and rotation curves, and five-step *decoder* traversals from +3 to −3 using a common reference image. Columns are ordered by ascending inferred Gaussian posterior variance (not variance of latent mean).

In [ ]:
import subprocess,sys
# ONE exact whole-dataset encoder evaluation per trained model; outputs all caption elements.
cmd=[sys.executable,'-m','src.visualize','--panel','both',
     '--data',str(DATA),'--output-root',str(RUNS),
     '--figures-dir',str(FIGURES),'--device','auto','--eval-batch-size','2048']
subprocess.run(cmd,check=True)


In [ ]:
display(Image(filename=str(FIGURES/'figure7_A.png')))
display(Image(filename=str(FIGURES/'figure7_B.png')))


## Evidence and limits
Compare specialization of Figure 7A against Figure 7B, verify actual KL/reconstruction values, and document mismatches. One seed does not prove generality. The paper shows illustrative outcomes, not exact numeric targets to hardcode.

In [ ]:
import json
with open(FIGURES/'figure7_metrics.json') as f:
    info=json.load(f)
for panel in ('A','B'):
    item=info[panel]
    print(f'Figure 7{panel}: β={item["beta"]}, seed={item["seed"]}, steps={item["steps"]}')
    print('Latent ids shown:',item['displayed_latent_ids_1_based'])
    print('Posterior variances:',[round(x,3) for x in item['mean_posterior_var_all_10']])
    print('Reconstruction:',item['reconstruction'])
# Persist finished images/metadata to Drive; never transfer live training writes to Drive.
BACKUP_FIGURES=BACKUP/'figures'
BACKUP_FIGURES.mkdir(parents=True,exist_ok=True)
for f in FIGURES.glob('figure7_*'):
    shutil.copy2(f,BACKUP_FIGURES/f.name)
print('Also backed up to',BACKUP_FIGURES)
print('Outputs:',*[str(p) for p in sorted(FIGURES.iterdir())],sep='\n  ')
